Undersök Datasetet

In [28]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

#Ladda in datasetet
df = pd.read_csv('Dataset/train.csv')
print("\n=== Sammanfattning av Data ===")
print(df.info())
print("\n=== Datatyper och unika värden per kolumn ===")
for col in df.columns:
    print(f"{col}: {df[col].dtype}, {df[col].nunique()} unika värden")
print("--------------------------------")
print("Dubblikater i Data:", df.duplicated().sum())
print("Missing value detection:",df.isnull().sum().sum())





=== Sammanfattning av Data ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB
None

=== Datatyper och unika värden per kolumn ===
date: object, 1826 unika värden
store: int64, 10 unika värden
item: int64, 50 unika värden
sales: int64, 213 unika värden
--------------------------------
Dubblikater i Data: 0
Missing value detection: 0


### Minimal Data Quality Checks (~80% täckning)
- Kolumner: obligatoriska kolumner finns (`date`, `store`, `item`, `sales`).
- Typkonvertering: `date`→datetime (upptäckt format), `store`/`item`/`sales`→int.
- Nulls: inga saknade värden; rapportera per kolumn.
- Nyckelunikhet: inga dubletter på (`date`,`store`,`item`).
- Per-dag fullständighet: radräkning per datum == `n_stores * n_items`.
- Volym sanity: total radräkning ≈ `n_dates * n_stores * n_items`.
- Sales rimlighet: `sales >= 0` och flagga extrema outliers (> 99.9:e percentil × 1.5).
Dessa regler är data-drivna (härleder förväntningar från aktuell data) för låg underhållskostnad.

In [32]:
# Kör minimala datakontroller
import pandas as pd

def run_min_checks(df: pd.DataFrame, date_format: str = "%Y-%m-%d"):
    issues = {}
    # Kolumner
    required = ["date", "store", "item", "sales"]
    missing_cols = [c for c in required if c not in df.columns]
    issues["missing_columns"] = missing_cols

    # Typkonvertering (tålig) + kopia
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], format=date_format, errors="coerce")
    d["store"] = pd.to_numeric(d["store"], errors="coerce")
    d["item"]  = pd.to_numeric(d["item"],  errors="coerce")
    d["sales"] = pd.to_numeric(d["sales"], errors="coerce")

    type_fail = {
        "date": int(d["date"].isna().sum()),
        "store": int(d["store"].isna().sum()),
        "item":  int(d["item" ].isna().sum()),
        "sales": int(d["sales"].isna().sum()),
    }
    issues["type_conversion_nulls"] = type_fail

    # Nulls (efter konvertering)
    nulls = d.isna().sum().to_dict()
    issues["nulls_per_column"] = {k:int(v) for k,v in nulls.items()}

    # Nyckelunikhet
    dup_cnt = int(d.duplicated(["date","store","item"]).sum())
    issues["key_duplicates"] = dup_cnt

    # Härled förväntad cardinalitet
    n_stores = int(d["store"].nunique())
    n_items  = int(d["item" ].nunique())
    n_dates  = int(d["date" ].nunique())
    issues["cardinality"] = {"stores": n_stores, "items": n_items, "dates": n_dates}

    # Per-dag fullständighet
    expected_per_day = n_stores * n_items
    per_day = d.groupby("date").size()
    bad_days = per_day[per_day != expected_per_day]
    issues["bad_days_count"] = int(bad_days.size)
    issues["bad_days_examples"] = [str(idx.date()) for idx in bad_days.index[:5]]

    # Volym sanity
    total_expected = expected_per_day * n_dates
    issues["total_rows"] = int(len(d))
    issues["total_expected"] = int(total_expected)
    issues["total_match"] = (len(d) == total_expected)

    # Sales rimlighet
    nonneg_fail = int((d["sales"] < 0).sum())
    upper = float(d["sales"].quantile(0.999) * 1.5)
    outliers_high = int((d["sales"] > upper).sum())
    issues["sales_checks"] = {
        "nonneg_fail": nonneg_fail,
        "upper_bound": upper,
        "outliers_high": outliers_high,
    }

    # Print sammanfattning
    print("=== Minimal Data Quality Checks ===")
    print("Missing columns:", issues["missing_columns"])
    print("Type conversion nulls:", issues["type_conversion_nulls"])
    print("Nulls per column:", issues["nulls_per_column"])
    print("Key duplicates:", issues["key_duplicates"])
    print("Cardinality:", issues["cardinality"])
    print(f"Per-day expected {expected_per_day}, bad days:", issues["bad_days_count"])
    print("Bad day examples:", issues["bad_days_examples"])
    print("Total rows:", issues["total_rows"], "| Total expected:", issues["total_expected"], "| Match:", issues["total_match"])
    print("Sales checks:", issues["sales_checks"])
    return issues

checks = run_min_checks(df, date_format="%Y-%m-%d")

=== Minimal Data Quality Checks ===
Missing columns: []
Type conversion nulls: {'date': 0, 'store': 0, 'item': 0, 'sales': 0}
Nulls per column: {'date': 0, 'store': 0, 'item': 0, 'sales': 0}
Key duplicates: 0
Cardinality: {'stores': 10, 'items': 50, 'dates': 1826}
Per-day expected 500, bad days: 0
Bad day examples: []
Total rows: 913000 | Total expected: 913000 | Match: True
Sales checks: {'nonneg_fail': 0, 'upper_bound': 246.0, 'outliers_high': 0}
